# Week 6 guided practice
CML3113. Follow the lecturer’s PowerPoint in this order. This notebook contains worked examples, not your graded upload.

Extract the student ZIP and keep its files together. For file-reading examples, upload the supplied CSV into the Colab runtime before running the file-reading example. From the file-reading lesson onward, worked table examples use the supplied CSV. Earlier syntax examples introduce Python before file access.

**A6 and Quiz 6 are waived. This is ungraded practice. No submission is required.**


## Today: clean, check and explain
*PowerPoint slide 2*

Inspect the CSV → remove duplicate observations → keep valid rows → fill missing temperatures → summarise → plot.

A6 and Quiz 6 are waived for CML3113. This lesson is ungraded practice; no upload or quiz response is required.




## The supplied file
*PowerPoint slide 3*

Extract this week’s student ZIP. Upload engineering_data.csv into Colab using the Files panel, or keep it beside the notebook locally.

The examples below read this file. Run the cells in slide order. Keep the original CSV unchanged.




## Operations for the next example
*PowerPoint slide 4*

Import and short names: import pandas as pd loads pandas and gives it the short name pd. pd.read_csv refers to a function in that library.

Reading a CSV: pd.read_csv("filename.csv") reads the named file into a DataFrame: a table with labelled columns and row labels. The file must exist in the working folder.

Counting dimensions: table.shape gives (number of rows, number of columns). shape is a property, so it has no parentheses.




## Operations for the next example
*PowerPoint slide 5*

Column data types: table.dtypes reports each column’s stored type. int64 means integers, float64 means decimal numbers, and object often contains text. Numeric blanks commonly appear as NaN.

A preview of rows: table.head() returns the first five rows. It leaves the table unchanged. A preview is not the entire dataset.




## Two names for two tables
*PowerPoint slide 6*

raw_data: the original table read from the CSV.

cleaned_data: a separate working table that we will change.




## Loading the supplied observations
*PowerPoint slide 7*



Run this example on the supplied engineering_data.csv.


In [ ]:
import pandas as pd
raw_data = pd.read_csv("engineering_data.csv")
print(raw_data.head())
print(raw_data.shape)
print(raw_data.dtypes)


**Example output** (timings and display formatting may vary)

```text
record_id equipment_id  load_pct  ambient_c  target
0          1           E1     59.15      31.71  14.001
1          2           E2     77.46      29.97  15.147
2          3           E3     69.60      19.87  13.005
3          4           E4     75.98      22.29     NaN
4          5           E5     57.62      23.40  12.554
(91, 5)
record_id         int64
equipment_id     object
load_pct        float64
ambient_c       float64
target          float64
dtype: object
```


## EDA: understand the original observations
*PowerPoint slide 8*

EDA means exploratory data analysis: inspect what the columns mean, their types, missing values, ranges and patterns before choosing cleaning actions.

record_id identifies an observation; equipment_id identifies its source. load_pct is %, ambient_c is °C, and target is crusher energy (kWh/t).




## Check types and recorded values
*PowerPoint slide 9*

table.info() prints column types and non-missing counts. Call it directly; do not wrap it in print.

table[numeric_columns].describe() summarises selected numeric measurements: count, mean, standard deviation, minimum, quartiles and maximum.




## Inspect structure and numerical ranges
*PowerPoint slide 10*



Run after the preceding example using the supplied CSV.


In [ ]:
raw_data.info()
numeric_columns = ["load_pct", "ambient_c", "target"]
print(raw_data[numeric_columns].describe())


**Example output** (timings and display formatting may vary)

```text
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91 entries, 0 to 90
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   record_id     91 non-null     int64  
 1   equipment_id  91 non-null     object 
 2   load_pct      91 non-null     float64
 3   ambient_c     90 non-null     float64
 4   target        90 non-null     float64
dtypes: float64(3), int64(1), object(1)
memory usage: 3.7+ KB
        load_pct  ambient_c     target
count  91.000000  90.000000  90.000000
mean   56.027692  26.467444  12.115478
std    23.026181   4.641630   2.874421
min    -5.000000  18.350000   6.625000
25%    35.430000  22.562500   9.637000
50%    59.150000  26.045000  12.432000
75%    76.330000  30.772500  14.122500
max    93.150000  34.970000  17.295000
```


## Check which equipment is represented
*PowerPoint slide 11*

value_counts() counts rows for each label. dropna=False also shows missing labels.

Counts can reveal missing groups or unequal coverage. Repeated equipment IDs are expected; repeated record IDs need separate inspection.




## Count observations per equipment
*PowerPoint slide 12*



Run after the preceding example using the supplied CSV.


In [ ]:
equipment_counts = raw_data["equipment_id"].value_counts(
    dropna=False)
print(equipment_counts)


**Example output** (timings and display formatting may vary)

```text
equipment_id
E1    16
E2    15
E3    15
E4    15
E5    15
E6    15
Name: count, dtype: int64
```


## Preview the raw distribution
*PowerPoint slide 13*

A histogram groups numeric values into intervals and counts observations in each interval. bins=10 requests ten intervals.

plt.subplots() gives a figure and its axes. ax selects where the chart is drawn. Missing numeric values are omitted from the plot.




## Plot the target before cleaning
*PowerPoint slide 14*



Run after the preceding example using the supplied CSV.


In [ ]:
import matplotlib.pyplot as plt
raw_figure, raw_axes = plt.subplots()
raw_data["target"].plot.hist(bins=10, ax=raw_axes)
raw_axes.set_xlabel("crusher energy (kWh/t)")
raw_axes.set_ylabel("Observation count")
raw_axes.set_title("Before cleaning")
plt.show()


**Example output** (timings and display formatting may vary)

```text
This cell stores values for the next example; it does not print a result.
```


## Inspect a relationship before cleaning
*PowerPoint slide 15*

A scatter plot places one observation at its load and target coordinates. x and y name the two columns; ax selects the plotting area.

Look for gaps, unusual points and possible trends. A visible association does not show that load caused the target to change.




## Plot raw load against the target
*PowerPoint slide 16*



Run after the preceding example using the supplied CSV.


In [ ]:
raw_scatter_figure, raw_scatter_axes = plt.subplots()
raw_data.plot.scatter(x="load_pct", y="target", ax=raw_scatter_axes)
raw_scatter_axes.set_xlabel("Load (%)")
raw_scatter_axes.set_ylabel("crusher energy (kWh/t)")
raw_scatter_axes.set_title("Before cleaning")
plt.show()


**Example output** (timings and display formatting may vary)

```text
This cell stores values for the next example; it does not print a result.
```


## A missing value is not zero
*PowerPoint slide 17*

NaN means a value is missing. A measured zero is still a recorded value.

isna() asks “is this cell missing?” It returns True for yes and False for no.

.sum() counts True answers: True counts as 1 and False as 0.




## Count missing cells in each column
*PowerPoint slide 18*



Use the supplied engineering_data.csv. Run cells in order.


In [ ]:
print(raw_data.isna().sum())


**Example output** (timings and display formatting may vary)

```text
record_id       0
equipment_id    0
load_pct        0
ambient_c       1
target          1
dtype: int64
```


## Find a repeated observation
*PowerPoint slide 19*

record_id identifies one observation. equipment_id identifies the equipment and may repeat.

duplicated("record_id") marks later copies True. The first occurrence is False.




## Count repeated record IDs
*PowerPoint slide 20*



Use the supplied engineering_data.csv. Run cells in order.


In [ ]:
is_repeated_record = raw_data.duplicated("record_id")
print(is_repeated_record)
print(is_repeated_record.sum())


**Example output** (timings and display formatting may vary)

```text
0     False
1     False
2     False
3     False
4     False
      ...  
86    False
87    False
88    False
89    False
90     True
Length: 91, dtype: bool
1
```


## Remove the later copy; keep the original
*PowerPoint slide 21*

drop_duplicates("record_id") keeps the first occurrence of each record ID.

.copy() makes an independent working table. len(table) counts its rows.




## Create the working table
*PowerPoint slide 22*



Use the supplied engineering_data.csv. Run cells in order.


In [ ]:
cleaned_data = raw_data.drop_duplicates("record_id").copy()
print(len(raw_data))
print(len(cleaned_data))


**Example output** (timings and display formatting may vary)

```text
91
90
```


## Question 1: is the target recorded?
*PowerPoint slide 23*

cleaned_data["target"] selects the target column.

.notna() asks whether each value is present. It returns True for a recorded value and False for a missing value.




## Look at the values, then their answers
*PowerPoint slide 24*



Use the supplied engineering_data.csv. Run cells in order.


In [ ]:
target_is_recorded = cleaned_data["target"].notna()
print(cleaned_data["target"].head())
print(target_is_recorded.head())


**Example output** (timings and display formatting may vary)

```text
0    14.001
1    15.147
2    13.005
3       NaN
4    12.554
Name: target, dtype: float64
0     True
1     True
2     True
3    False
4     True
Name: target, dtype: bool
```


## Use the answers to select rows
*PowerPoint slide 25*

cleaned_data[target_is_recorded] keeps rows whose answer is True. Rows answering False are left out of the returned table.

This True/False selection is called a Boolean mask. The variable name does not need to end in _mask.




## Try one condition on its own
*PowerPoint slide 26*



Use the supplied engineering_data.csv. Run cells in order.


In [ ]:
rows_with_target = cleaned_data[target_is_recorded].copy()
print(len(cleaned_data))
print(len(rows_with_target))


**Example output** (timings and display formatting may vary)

```text
90
89
```


## Question 2: is the load valid?
*PowerPoint slide 27*

load_pct is a percentage. For this exercise the valid range is 0 through 100, including both endpoints.

.between(0, 100) answers True inside that range and False outside it.




## Inspect the load answers
*PowerPoint slide 28*



Use the supplied engineering_data.csv. Run cells in order.


In [ ]:
load_is_valid = cleaned_data["load_pct"].between(0, 100)
print(cleaned_data["load_pct"].head(10))
print(load_is_valid.head(10))


**Example output** (timings and display formatting may vary)

```text
0    59.15
1    77.46
2    69.60
3    75.98
4    57.62
5    50.12
6    24.66
7    72.92
8    -5.00
9    67.91
Name: load_pct, dtype: float64
0     True
1     True
2     True
3     True
4     True
5     True
6     True
7     True
8    False
9     True
Name: load_pct, dtype: bool
```


## Keep a row only when both answers are True
*PowerPoint slide 29*

& means “and” for two True/False columns.

True & True → True. Any pair containing False → False.

Our rule: a recorded target AND a valid load.




## Which observations will we keep?
*PowerPoint slide 30*



| Record ID | Target recorded? | Load valid? | Keep? |
| --- | --- | --- | --- |
| 1 | True | True | Yes |
| 4 | False | True | No |
| 9 | True | False | No |

A row stays only when both checks pass. These records come from the supplied CSV.


## Combine the answers, then keep those rows
*PowerPoint slide 31*



Use the supplied engineering_data.csv. Run cells in order.


In [ ]:
rows_to_keep = target_is_recorded & load_is_valid
print(rows_to_keep.sum())
cleaned_data = cleaned_data[rows_to_keep].copy()
print(len(cleaned_data))


**Example output** (timings and display formatting may vary)

```text
88
88
```


## Fill only missing ambient temperatures
*PowerPoint slide 32*

ambient_c is temperature in °C. .median() finds the middle of the recorded values in the retained rows.

.fillna(value) replaces only missing entries. Assign the result back to the ambient_c column.




## Fill missing ambient temperatures
*PowerPoint slide 33*



Run this example on the supplied engineering_data.csv.


In [ ]:
median_ambient_c = cleaned_data["ambient_c"].median()
cleaned_data["ambient_c"] = (
    cleaned_data["ambient_c"].fillna(median_ambient_c))
print(round(float(median_ambient_c), 3))
print(cleaned_data.isna().sum())


**Example output** (timings and display formatting may vary)

```text
26.07
record_id       0
equipment_id    0
load_pct        0
ambient_c       0
target          0
dtype: int64
```


## Validate the result before interpreting charts
*PowerPoint slide 34*

Count original and retained rows. Check duplicate record IDs, missing values and valid loads again.

.all() answers True only when every row passes the condition. A check confirms what the data contain; a table named cleaned_data is not automatically correct.




## Check the cleaned table
*PowerPoint slide 35*



Run after the preceding example using the supplied CSV.


In [ ]:
print("Rows before:", len(raw_data))
print("Rows after:", len(cleaned_data))
print("Duplicate IDs:", cleaned_data.duplicated("record_id").sum())
print(cleaned_data.isna().sum())
print("Loads valid:", cleaned_data["load_pct"].between(0, 100).all())


**Example output** (timings and display formatting may vary)

```text
Rows before: 91
Rows after: 88
Duplicate IDs: 0
record_id       0
equipment_id    0
load_pct        0
ambient_c       0
target          0
dtype: int64
Loads valid: True
```


## Summaries and grouped results
*PowerPoint slide 36*

mean, min and max summarise a numeric column. groupby collects rows with the same equipment label before calculating.

Use the cleaned table for summaries and charts.




## Operations for the next example
*PowerPoint slide 37*

Mean of recorded values: series.mean() adds known values and divides by their count. pandas skips missing values by default. The result has the same unit as the original column.

Several summaries at once: series.agg(["mean", "min", "max"]) applies those three named calculations and returns a labelled Series of results. min and max give the lowest and highest recorded values.

Grouping rows: table.groupby("label")["value"] groups the value column by matching labels. Apply mean() or sum() to calculate one result per group. Missing group labels are excluded by default.




## Summaries of the cleaned CSV
*PowerPoint slide 38*



Run this example on the supplied engineering_data.csv.


In [ ]:
target_summary = cleaned_data["target"].agg(["mean", "min", "max"])
mean_target_by_equipment = (
    cleaned_data.groupby("equipment_id")["target"].mean())
print(target_summary)
print(mean_target_by_equipment.round(3))


**Example output** (timings and display formatting may vary)

```text
mean    12.073682
min      6.625000
max     17.295000
Name: target, dtype: float64
equipment_id
E1    12.251
E2    12.504
E3    12.131
E4    13.248
E5    10.651
E6    11.740
Name: target, dtype: float64
```


## Choosing a chart for the question
*PowerPoint slide 39*


See the illustration on this PowerPoint slide.

This week uses histograms, scatter plots and bar charts. The other chart types are reference examples.


## Charts and units
*PowerPoint slide 40*

A scatter plot compares two numerical columns. A histogram counts observations in value intervals. A bar chart compares group summaries.

plt.subplots creates a Figure and plotting Axes. Use ax= to choose where pandas draws, then label each axis.




## Operations for the next example
*PowerPoint slide 41*

plt.subplots() returns two objects: a figure (the whole image) and axes (the plotting area). Store them as scatter_figure and scatter_axes.

The ax= argument tells pandas where to draw: ax=scatter_axes uses that plotting area. Keep the library keyword ax unchanged.




## Load and crusher energy demand
*PowerPoint slide 42*



Run this example on the supplied engineering_data.csv.


In [ ]:
import matplotlib.pyplot as plt
scatter_figure, scatter_axes = plt.subplots()
cleaned_data.plot.scatter(
    x="load_pct", y="target", ax=scatter_axes)
scatter_axes.set_xlabel("Load (%)")
scatter_axes.set_ylabel("Crusher energy (kWh/t)")
plt.show()


**Example output** (timings and display formatting may vary)

```text
This cell stores values for the next example; it does not print a result.
```


## Operations for the next example
*PowerPoint slide 43*

A bar for each labelled value: series.plot.bar(ax=ax) draws a bar for each Series label. When the Series contains group means, bar heights represent those means.

Histogram intervals: series.plot.hist(bins=10, ax=ax) groups known values into ten value intervals and counts observations in each. Bar heights are counts, not the original measurement values.




## Mean crusher energy for each equipment group
*PowerPoint slide 44*



Run this example on the supplied engineering_data.csv.


In [ ]:
bar_figure, bar_axes = plt.subplots()
mean_target_by_equipment.plot.bar(ax=bar_axes)
bar_axes.set_xlabel("Equipment ID")
bar_axes.set_ylabel("Mean crusher energy (kWh/t)")
plt.show()


**Example output** (timings and display formatting may vary)

```text
This cell stores values for the next example; it does not print a result.
```


## Distribution of crusher energy readings
*PowerPoint slide 45*



Run this example on the supplied engineering_data.csv.


In [ ]:
histogram_figure, histogram_axes = plt.subplots()
cleaned_data["target"].plot.hist(
    bins=10, ax=histogram_axes)
histogram_axes.set_xlabel("Crusher energy (kWh/t)")
histogram_axes.set_ylabel("Observation count")
plt.show()


**Example output** (timings and display formatting may vary)

```text
This cell stores values for the next example; it does not print a result.
```


## Optional extension: check numeric text
*PowerPoint slide 46*

The main cleaning and charting walkthrough is complete. Continue here only if time permits.

Sometimes a numeric column contains text that cannot be converted to a number. The following check distinguishes that problem from an existing blank.




## Operations for the next example
*PowerPoint slide 47*

Converting numeric text: pd.to_numeric(column, errors="coerce") converts numeric values or text. Unparseable text becomes missing. Compare original and converted values so existing gaps are not counted as conversion failures.




## Check numeric conversion of ambient temperature
*PowerPoint slide 48*



Run this example on the supplied engineering_data.csv.


In [ ]:
ambient_numbers = pd.to_numeric(
    raw_data["ambient_c"], errors="coerce")
value_was_recorded = raw_data["ambient_c"].notna()
conversion_failed = value_was_recorded & ambient_numbers.isna()
print(int(conversion_failed.sum()))
print(raw_data[conversion_failed].head())


**Example output** (timings and display formatting may vary)

```text
0
Empty DataFrame
Columns: [record_id, equipment_id, load_pct, ambient_c, target]
Index: []
```


## Optional extension: put steps in a function
*PowerPoint slide 49*

A function gives a reusable calculation a name. Its body runs when the function is called.

The next example calculates mean load for each equipment group; it is practice, not an A6 requirement.




## Operations for the next example
*PowerPoint slide 50*

Returning a dictionary: series.to_dict() maps Series labels to values. table.to_dict("records") instead returns a list of row dictionaries. Choose the form specified in the example.

A table from Python values: pd.DataFrame(records) builds a table from row dictionaries. With a dictionary of column lists, keys become column names and each list supplies that column’s values.




## A function called with the supplied records
*PowerPoint slide 51*



Run this example on the supplied engineering_data.csv.


In [ ]:
def mean_load_by_equipment(equipment_records):
    equipment_data = pd.DataFrame(equipment_records)
    mean_load_by_group = (
        equipment_data.groupby("equipment_id")["load_pct"].mean())
    return mean_load_by_group.to_dict()
raw_records = raw_data.to_dict("records")
print(mean_load_by_equipment(raw_records))


**Example output** (timings and display formatting may vary)

```text
{'E1': 54.9475, 'E2': 61.326, 'E3': 53.46933333333333, 'E4': 66.854, 'E5': 48.10133333333333, 'E6': 51.54}
```


## Save your practice
A6 and Quiz 6 are waived. No submission is required. Restart and run all cells, then save in Drive. Continue with the Test 1 revision PDF and matching practice CSV in a new notebook.
